## Row-Level and Column-Level Security

### Load and Prepare Data

In [ ]:
%sql
CREATE VOLUME IF NOT EXISTS spark_lab

In [ ]:
import requests
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Define the current catalog and volume
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]
volume_base = f"/Volumes/{catalog_name}/default/spark_lab"

# Define schema for sales data
schema = StructType([
    StructField("SalesOrderNumber", StringType()),
    StructField("SalesOrderLineNumber", IntegerType()),
    StructField("OrderDate", DateType()),
    StructField("CustomerName", StringType()),
    StructField("Email", StringType()),
    StructField("Item", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("UnitPrice", FloatType()),
    StructField("Tax", FloatType())
])

# Load data from 2019, 2020, 2021 CSV files
df = spark.read.schema(schema) \
    .option("header", "true") \
    .csv(f'{volume_base}/*_edited.csv')

display(df.limit(10))

### Store Raw Data as Delta Lake Table

In [ ]:
# Store the consolidated sales data as a Delta table
raw_data_path = f'/Volumes/{catalog_name}/default/spark_lab/delta/sales_raw'
df.write.format('delta').mode('overwrite').save(raw_data_path)

# Register as SQL table
df.write.format('delta').mode('overwrite').saveAsTable('default.sales_raw')

print("Raw data stored successfully")
print(f"Total records: {df.count()}")

### Creating a User-Defined Function (UDF) for Column Masking

In [ ]:
%sql
CREATE OR REPLACE FUNCTION email_mask(Email STRING)
RETURN CASE 
  WHEN current_user() IN (
    'ENTER_EMAIL_1_HERE',
    'ENTER_EMAIL_2_HERE'
  )
  THEN "*****@*****.***"
  ELSE Email
END;

### Apply the Masking Function to the Table

In [ ]:
%sql
ALTER TABLE default.sales_raw
  ALTER COLUMN Email 
  SET MASK email_mask;

### Query the Table to See the Masked Data

In [ ]:
%sql
SELECT * FROM default.sales_raw

### UDF for Row-Level Security

In [ ]:
%sql
CREATE OR REPLACE FUNCTION customer_filter(CustomerName STRING)
RETURN IF(
  current_user() = 'admin@gmail.com',
  true,
  CustomerName = 'Mariah Foster'
);

### Apply Row Level Security to the Table

In [ ]:
%sql
ALTER TABLE default.sales_raw 
SET ROW FILTER customer_filter ON (CustomerName);

### Query the Masked Data

In [ ]:
%sql
SELECT * FROM default.sales_raw

In [ ]:
%sql
SELECT * FROM default.sales_raw
WHERE CustomerName = 'Armando Dominguez'

### Drop the UDFs and Remove Security Policies

In [ ]:
ALTER TABLE default.sales_raw 
DROP ROW FILTER;

In [ ]:
ALTER TABLE default.sales_raw 
ALTER COLUMN Email 
DROP MASK;